<a href="https://colab.research.google.com/github/mfachriridwan/claude/blob/main/Research_Digitaltwin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd
CATS=["sisa_makanan","daun_kering","kertas_kardus","kayu","plastik","logam","kain","karet_kulit","sterofoam","b3","elektronik","lain_lain"]
MAP={"F_waste":["sisa_makanan"],"Gard_waste":["daun_kering","kayu"],"Pap_Card":["kertas_kardus"],
     "Composites":["kain","karet_kulit","elektronik"],"Plastic":["plastik","sterofoam"],
     "Metal":["logam"],"Glass":["kaca"],"Inert":["b3","lain_lain"]}
FR=list(MAP)
CITIES=[
 dict(city="Kota Padang",year_base=2023,growth=0.0126,horizon=2044,tpd_dom=620.73,tpd_nd=26.84,
      dom=[43.95,15.68,8.85,0.42,19.84,1.30,1.80,0.27,0.22,0.18,1.09,6.41],
      nd =[51.56,25.78,6.34,0.26,9.61,1.36,0.93,0.26,0.54,0.33,0.17,2.85]),
 dict(city="Kota Denpasar",year_base=2023,growth=0.00126,horizon=2044,tpd_dom=114.35,tpd_nd=693.79,
      dom=[17.32,58.32,10.84,0,7.33,0.38,0.66,1.11,0.10,0,0,3.94],
      nd =[27.75,40.91,7.84,0,11.72,3.33,1.11,3.41,0.30,0,0,3.62]),
 dict(city="Kota Depok",year_base=2023,growth=0.0179,horizon=2044,tpd_dom=759.47,tpd_nd=506.32,
      dom=[40, 0, 10, 0, 14, 1, 2, 1, 0, 0, 0, 32],
      nd =[44, 0, 12, 0, 18, 0, 0, 1, 0, 1, 0, 24]),
 dict(city="Kab. Bogor", year_base=2025, growth=0.0244, horizon=2045,
     tpd_dom=2942.0, tpd_nd=0.0,   # komposisi tunggal Gambar 3.7 -> dom = nd
     dom=[55.30, 2.26, 7.60, 0, 16.57, 1.13, 2.70, 0.12, 0, 0, 0, 14.31],
     nd =[55.30, 2.26, 7.60, 0, 16.57, 1.13, 2.70, 0.12, 0, 0, 0, 14.31]),
 # tambah kota lain: salin satu baris dict di atas
]
to8=lambda v:(lambda c:[sum(c.get(x,0) for x in MAP[f]) for f in FR])(dict(zip(CATS,v)))
def macro(k):
    w=k["tpd_dom"]/(k["tpd_dom"]+k["tpd_nd"]); d,n=to8(k["dom"]),to8(k["nd"])
    return {**{x:k[x] for x in("city","year_base","growth","horizon")},
            "timbulan_tpd":round(k["tpd_dom"]+k["tpd_nd"],2),
            **{f:round(w*d[i]+(1-w)*n[i],2) for i,f in enumerate(FR)}}
tabel=pd.DataFrame([macro(k) for k in CITIES]); tabel

try:                                    # tampilan interaktif kalau di Colab
    from google.colab import data_table
    display(data_table.DataTable(tabel))
except Exception:
    print(tabel.to_string(index=False))
print("\n-> input_kota.csv tersimpan (", len(tabel), "kota )")

,city,year_base,growth,horizon,timbulan_tpd,F_waste,Gard_waste,Pap_Card,Composites,Plastic,Metal,Glass,Inert
0,Kota Padang,2023,0.01260,2044,647.57,44.27,16.51,8.75,3.09,19.65,1.30,0.0,6.45
1,Kota Denpasar,2023,0.00126,2044,808.14,26.27,43.37,8.26,4.13,11.37,2.91,0.0,3.67
2,Kota Depok,2023,0.01790,2044,1265.79,41.60,0.00,10.80,2.20,15.60,0.60,0.0,29.20
3,Kab. Bogor,2025,0.02440,2045,2942.00,55.30,2.26,7.60,2.82,16.57,1.13,0.0,14.31



-> input_kota.csv tersimpan ( 4 kota )


In [40]:
import numpy as np, pandas as pd
FR=["F_waste","Gard_waste","Pap_Card","Composites","Plastic","Metal","Glass","Inert"]
DOC=np.array([.15,.20,.40,.24,0,0,0,0]); CV=np.array([.15,.15,.15,.35,.15,.35,.35,.35])
DOCf,MCF,F,OX,GWP=.5,.8,.5,0.,28; REL=[.20,.25,.20,.05]   # rel DOCf,MCF,F,timbulan
def emisi(r):
    s=np.array([r[f] for f in FR],float); s/=s.sum()
    doc=s@DOC; k=MCF*doc*DOCf*F*(16/12)*(1-OX)*GWP
    rel=np.hypot(np.sqrt(((DOC*s*CV)**2).sum())/doc, np.linalg.norm(REL))
    yy=np.arange(int(r.year_base),int(r.horizon)+1)
    ton=r.timbulan_tpd*365*(1+r.growth)**(yy-int(r.year_base)); e=ton*k
    return pd.DataFrame({"city":r.city,"tahun":yy,"timbulan_ton":ton.round(),"DOC":round(doc,4),
        "EF_kg_per_ton":round(k*1000,1),"CO2e_ton":e.round(),"u_rel_%":round(rel*100,1),
        "CO2e_lo":(e*(1-rel)).round(),"CO2e_hi":(e*(1+rel)).round()})
detail=pd.concat([emisi(r) for _,r in tabel.iterrows()],ignore_index=True)
ringkas=detail.groupby("city").agg(DOC=("DOC","first"),EF=("EF_kg_per_ton","first"),
    CO2e_awal=("CO2e_ton","first"),CO2e_akhir=("CO2e_ton","last"),u_rel=("u_rel_%","first")).reset_index()
ringkas

,city,DOC,EF,CO2e_awal,CO2e_akhir,u_rel
0,Kab. Bogor,0.1247,930.7,999439.0,1618630.0,39.6
1,Kota Denpasar,0.1691,1262.8,372502.0,382484.0,39.2
2,Kota Depok,0.1109,827.9,382503.0,555190.0,39.5
3,Kota Padang,0.1418,1058.9,250277.0,325550.0,39.1


In [38]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=ringkas)

https://docs.google.com/spreadsheets/d/1qU0VC81c5JqqntliylzTp02UAR-HMbDLQhCoOZxvpQY/edit#gid=0


In [44]:
display(ringkas.round(2))

,city,DOC,EF,CO2e_awal,CO2e_akhir,u_rel,CO2e_increase_percent
0,Kab. Bogor,0.12,930.7,999439.0,1618630.0,39.6,61.95
1,Kota Denpasar,0.17,1262.8,372502.0,382484.0,39.2,2.68
2,Kota Depok,0.11,827.9,382503.0,555190.0,39.5,45.15
3,Kota Padang,0.14,1058.9,250277.0,325550.0,39.1,30.08


In [57]:
import numpy as np, pandas as pd

FR=["F_waste","Gard_waste","Pap_Card","Composites","Plastic","Metal","Glass","Inert"]
DOC=np.array([.15,.20,.40,.24,0,0,0,0]); CV=np.array([.15,.15,.15,.35,.15,.35,.35,.35])
REL=np.array([.20,.25,.20,.05]); NAMES=["DOCf","MCF","F","Timbulan"]     # rel DOCf,MCF,F,timbulan

row=tabel.iloc[0]; s=np.array([row[f] for f in FR],float); s/=s.sum()    # kota pertama (Padang)
urel=lambda cv=CV,rel=REL: np.hypot(np.sqrt(((DOC*s*cv)**2).sum())/(s@DOC), np.linalg.norm(rel))*100

base=urel(); res=[]
for i,n in enumerate(NAMES):                                             # tiap parameter ±10%
    for d in (.1,-.1):
        r=REL.copy(); r[i]*=1+d
        res.append((f"{n}_uncertainty", f"{d:+.0%}", urel(rel=r)))
for d in (.1,-.1):                                                       # semua CV di-skala ±10%
    res.append(("All_CVs_scaled", f"{d:+.0%}", urel(cv=CV*(1+d))))

df=pd.DataFrame(res, columns=["Parameter","Direction","New u_rel (%)"])
df["Change from Base (%)"]=df["New u_rel (%)"]-base
df=df.reindex(df["Change from Base (%)"].abs().sort_values(ascending=False).index)
print(f"Base u_rel {row['city']}: {base:.2f}%\n"); display(df.round(2))

Base u_rel Kota Padang: 39.10%



,Parameter,Direction,New u_rel (%),Change from Base (%)
2,MCF_uncertainty,+10%,40.74,1.64
3,MCF_uncertainty,-10%,37.55,-1.55
4,F_uncertainty,+10%,40.16,1.06
0,DOCf_uncertainty,+10%,40.16,1.06
1,DOCf_uncertainty,-10%,38.11,-0.98
5,F_uncertainty,-10%,38.11,-0.98
8,All_CVs_scaled,+10%,39.31,0.21
9,All_CVs_scaled,-10%,38.91,-0.19
6,Timbulan_uncertainty,+10%,39.16,0.07
7,Timbulan_uncertainty,-10%,39.04,-0.06
